# 精确 f696 Stage1 去 beta 硬上限：0bp 单账户消融

用户授权研究去除原动作层 beta 上限。固定 f696 五输出、b472 动作模型、原 Qlib 原价/分红/费用来源，唯一干预为风险规格中的防守、近期亏损、紧急 beta 限值均设 3.0，保持原风险状态时钟及 SOXL/现金结构约束。自身状态账户与 SOXX 同源比较；结果为历史复用诊断，不更改现役 SIM。

In [1]:
import os
os.environ["MLFLOW_ALLOW_FILE_STORE"]="true"
os.environ["MLFLOW_DISABLE_AGENT_HINT"]="1"
import contextlib,hashlib,json
from dataclasses import replace
from pathlib import Path
import numpy as np
import pandas as pd
import qlib
from qlib.workflow import R
from src.core.recorder_sources import load_verified_recorder_objects
from src.backtest.stage1_domain_heads_raw import replay_stage1_raw
from src.backtest.native_raw_corporate import native_raw_validate
from src.workflow.intraday_recovery_direct_allocation import IntradayRecoveryDirectAllocationSpec
from src.workflow.stage1_open_action_value import Stage1OpenActionValueSpec
from src.workflow.stage1_rolling126_shortfall import _analyze_account
from src.data.minute_intake import atomic_json
root=Path.cwd().resolve();card=root/"research/stage1_f696_no_beta_cap_ablation_20261029"
plan_bytes=(card/"plan.json").read_bytes();plan=json.loads(plan_bytes);digest=hashlib.sha256(plan_bytes).hexdigest()
assert plan["frozen"] and plan["execution_authorized"] and not (card/"run_state.json").exists()
sha=lambda path:hashlib.sha256(Path(path).read_bytes()).hexdigest()
assert sha(root/plan["provider_uri"]/"qlib_store_manifest.json")==plan["provider_manifest_sha256"]
assert sha(root/plan["release_path"])==plan["release_sha256"]
qlib.init(provider_uri=str(root/plan["provider_uri"]),region="us",kernels=1)
def source(role,names):
    spec=plan[role];source_card=root/spec["card"]
    binding=json.loads((source_card/"nas_binding.json").read_text())
    assert binding["evidence"]["sha256"]==spec["archive_sha256"]
    if "summary_sha256" in spec:assert sha(source_card/"summary.json")==spec["summary_sha256"]
    return load_verified_recorder_objects(source_card,expected_recorder_id=spec["recorder_id"],artifacts=names,expected_manifest_sha256=spec["inventory_sha256"])
needed=["inputs/frame","inputs/raw_points","inputs/schedule","inputs/controller","inputs/selection","inputs/recovery_spec","inputs/action_spec","inputs/native_provider_manifest","native/A_active_research_replay_0"]
domain,domain_receipt=source("source_domain",needed)
action,action_receipt=source("source_action",["inputs/source_targets"])
raw,raw_receipt=source("source_raw",["native/raw_SOXX_0"])
frame=domain["inputs/frame"].reset_index(drop=True).copy()
targets=action["inputs/source_targets"]["frozen_current"]
columns=["soxl_weight","soxx_weight","cash_weight","tail_probability","alpha_probability"]
assert len(frame)==plan["expected_sessions"]==944
assert targets.index.equals(pd.DatetimeIndex(frame.signal_date)) and list(targets)==columns
np.testing.assert_allclose(frame[columns].to_numpy(float),targets[columns].to_numpy(float),rtol=0,atol=1e-12)
reference=domain["native/A_active_research_replay_0"]
soxx=raw["native/raw_SOXX_0"]
assert reference["audit"]["status"]==soxx["audit"]["status"]=="passed"
for book in (reference,soxx):
    assert len(book["authority"])==944
    pd.testing.assert_frame_equal(book["authority"][["signal_date","trade_date","exit_date"]].reset_index(drop=True),frame[["signal_date","trade_date","exit_date"]].reset_index(drop=True))
provider_manifest=json.loads((root/plan["provider_uri"]/"qlib_store_manifest.json").read_text())
assert domain["inputs/native_provider_manifest"]==provider_manifest
recovery=IntradayRecoveryDirectAllocationSpec(root=root,**domain["inputs/recovery_spec"])
uncapped=replace(recovery,defensive_beta=3.0,trailing_loss_beta=3.0,emergency_beta=3.0)
action_spec=Stage1OpenActionValueSpec(root=root,**domain["inputs/action_spec"])
execution={"initial_cash":plan["initial_cash_usd"],"tariff":json.loads((root/plan["source_domain"]["card"]/"plan.json").read_text())["tariff"]}
# The provider source contract's tariff is pinned by the original domain plan.
assert execution["tariff"]==json.loads((root/plan["source_domain"]["card"]/"plan.json").read_text())["tariff"]
state={"plan_sha256":digest,"status":"native_attempt_started","native_account_attempts":1,"bootstrap_attempts":0,"new_model_fits":0}
atomic_json(state,card/"run_state.json")
with open("/tmp/stage1_f696_no_beta_cap_ablation_20261029.log","w") as log,contextlib.redirect_stdout(log),contextlib.redirect_stderr(log):
    path=replay_stage1_raw(frame,point_prices=domain["inputs/raw_points"],schedule=domain["inputs/schedule"],plan=execution,controller=domain["inputs/controller"],recovery_spec=uncapped,action_spec=action_spec,selection=domain["inputs/selection"],slippage_bps=0)
    assert len(path["replay"])==944 and len(path["points"])==3778
    assert np.allclose(path["trace"].current_beta_cap.to_numpy(float),3.0,rtol=0,atol=1e-12)
    book=native_raw_validate(path,provider_uri=root/plan["provider_uri"],point_prices=domain["inputs/raw_points"],schedule=domain["inputs/schedule"],tariff=execution["tariff"],cost=0,initial_cash=execution["initial_cash"])
    assert book["audit"]["status"]=="passed"
    state.update(status="native_complete",bootstrap_attempts=1)
    atomic_json(state,card/"run_state.json")
    sampled_dd=-100*float(path["replay"].monitored_drawdown.max())
    result=_analyze_account(account=book["authority"],reference_soxx=soxx["authority"],reference_stage1=reference["authority"],sampled_drawdown=sampled_dd,plan=plan)
assert len(result["annual"])==4 and len(result["quarter"])==16 and len(result["rolling"])==819
tracking=card/"qlib_recorder"
with R.start(experiment_name=plan["experiment"],uri=str(tracking)):
    recorder=R.get_recorder()
    recorder.save_objects(artifact_path="inputs",plan=plan,source_audits={"domain":domain_receipt,"action":action_receipt,"raw":raw_receipt})
    recorder.save_objects(artifact_path="sources",replay=(root/"src/backtest/stage1_domain_heads_raw.py").read_text(),risk=(root/"src/workflow/intraday_recovery_direct_allocation.py").read_text(),open_action=(root/"src/workflow/stage1_open_action_value.py").read_text())
    recorder.save_objects(artifact_path="paths",no_beta_cap_0=path)
    recorder.save_objects(artifact_path="native",no_beta_cap_0=book)
    recorder.save_objects(artifact_path="analysis",SOXX_metrics=result["SOXX_metrics"],Stage1_increment=result["Stage1_increment_metrics"],annual=result["annual"],quarter=result["quarter"],rolling_summary=result["rolling_summary"],uncertainty=result["uncertainty"],gates=result["gates"])
    recorder.save_objects(artifact_path="analysis/rolling",all_windows=result["rolling"])
    recorder.log_metrics(new_model_fits=0,native_accounts=1,SOXX_relative_wealth=result["SOXX_metrics"]["relative_wealth"],historical_0bp_pass=int(result["historical_0bp_gate_pass"]))
    rid,eid=recorder.id,str(recorder.experiment_id)
summary={"experiment":plan["experiment"],"plan_sha256":digest,"decision":"no_beta_cap_0bp_coarse_pass_needs_cost_and_risk_qualification" if result["historical_0bp_gate_pass"] else "not_promoted_fixed_f696_no_beta_cap_0bp","objective_achieved":False,"historical_research_passed":False,"historical_0bp_gate_pass":bool(result["historical_0bp_gate_pass"]),"SOXX_metrics":result["SOXX_metrics"],"Stage1_increment_metrics":result["Stage1_increment_metrics"],"rolling_summary":result["rolling_summary"],"uncertainty":result["uncertainty"],"gates":result["gates"],"risk_ablation_audit":{"cap_min":float(path["trace"].current_beta_cap.min()),"cap_max":float(path["trace"].current_beta_cap.max()),"risk_state_nonzero_days":int(path["trace"].risk_state.ne(0).sum()),"recovery_actions":int(path["trace"].recovery_action.sum()),"open_execute_days":int(path["trace"].execute_open_target.sum()),"sampled_max_drawdown_pct":sampled_dd},"new_model_fits":0,"new_native_accounts":1,"qualified_forward_days":0,"recorder_id":rid,"experiment_id":eid,"recorder_uri":str(tracking),"recorder_local_dir":str(tracking/eid/rid),"provider_uri":str(root/plan["provider_uri"]),"artifact_policy":{"policy":"slim_card_recorder_first","workflow_config":"workflow.yaml","recorder_only_artifacts":["inputs/*","sources/*","paths/*","native/*","analysis/*"]},"limitations":["The action model receives cap3 outside its original training domain; recovery becomes ineligible mechanically.","2023-2026 history has been reused; this is not independent validation.","Available sparse marks do not certify full-minute max drawdown.","No 2/5bp or forward evidence."]}
atomic_json(summary,card/"summary.json")
result["annual"].to_csv(card/"annual_metrics.csv",index=False)
result["quarter"].to_csv(card/"quarter_metrics.csv",index=False)
atomic_json({"recorder_id":rid,"experiment_id":eid,"plan_sha256":digest,"status":"completed","native_account_attempts":1,"bootstrap_attempts":1,"new_model_fits":0,"historical_0bp_gate_pass":bool(result["historical_0bp_gate_pass"])},card/"run_state.json")
print(summary["decision"],summary["SOXX_metrics"]["relative_wealth"],summary["risk_ablation_audit"],summary["rolling_summary"]["positive_fraction"])

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[10120:MainThread](2026-10-29 10:37:35,805) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[10120:MainThread](2026-10-29 10:37:35,806) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[10120:MainThread](2026-10-29 10:37:35,806) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/stage1_domain_heads_20261018')}


backtest loop:   0%|          | 0/3778 [00:00<?, ?it/s]

--- Logging error ---
Traceback (most recent call last):
  File "/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.12/site-packages/qlib/workflow/expm.py", line 391, in _get_exp
    raise MlflowException("No valid experiment has been found.")
mlflow.exceptions.MlflowException: No valid experiment has been found.

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.12/site-packages/qlib/workflow/expm.py", line 225, in _get_or_create_exp
    self._get_exp(experiment_id=experiment_id, experiment_name=experiment_name),
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.12/site-packages/qlib/workflow/expm.py", line 395, in _get_exp
    raise ValueError(
ValueError: No valid experiment has been found, please make sure the input experiment name is correct.

During handling of the above exception

not_promoted_fixed_f696_no_beta_cap_0bp 0.8829978781838212 {'cap_min': 3.0, 'cap_max': 3.0, 'risk_state_nonzero_days': 537, 'recovery_actions': 0, 'open_execute_days': 854, 'sampled_max_drawdown_pct': -47.24839386326095} 0.3601953601953602


--- Logging error ---
Traceback (most recent call last):
  File "/Users/rziiiii/.local/share/uv/python/cpython-3.12.12-macos-aarch64-none/lib/python3.12/logging/__init__.py", line 1163, in emit
    stream.write(msg + self.terminator)
ValueError: I/O operation on closed file.
Call stack:
  File "/Users/rziiiii/.local/share/uv/python/cpython-3.12.12-macos-aarch64-none/lib/python3.12/runpy.py", line 198, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/Users/rziiiii/.local/share/uv/python/cpython-3.12.12-macos-aarch64-none/lib/python3.12/runpy.py", line 88, in _run_code
    exec(code, run_globals)
  File "/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.12/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.12/site-packages/traitlets/config/application.py", line 1080, in launch_instance
    app.start()
  File "/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.12

## 中文结论

已执行精确f696五输出944/944提案核对、b472原动作模型及1本独立状态0bp Tiered原生Qlib账户；去除硬beta上限后cap全程3.0，风险状态仍递推，恢复动作0。相对原f696财富1.287，但相对SOXX仅0.883；可用收盘/监控最大回撤−46.82%/−47.25%，CVaR5−5.735%弱于SOXX−4.905%，完整年0/2超额，126日窗口295/819，区块下界−0.000715/日。所有0bp数值硬门失败，决定 `not_promoted_fixed_f696_no_beta_cap_0bp`；本版本不跑2/5bp，不改现役SIM。

逐年权重显示SOXL提高、现金下降；原风险状态非零353日，新账户537日。cap=3.0使b472模型输入越过训练域且恢复资格失效，不能将整条路径变化归为单一上限阈值的因果收益。2023–2026历史复用、完整分钟风险未认证；NAS工件上传失败，Recorder本地保留。详情见 `summary.json`、`annual_metrics.csv`、`quarter_metrics.csv`、`failure_attribution.json`。